# 03 · Modelos de ensamble en E1, E1+, E2 y E3

**TFM:** Predicción temprana del abandono con datos de matrícula y modelos de ensamble frente a reglas tradicionales.

**Propósito.** Responder P1 y P2 con la partición y la regla ya fijadas en `02_regla_base`.

**Protocolo**

- La partición es la de `resultados/idx_train.csv` y `idx_test.csv`. No se vuelve a sortear.
- SMOTE se ajusta solo dentro de cada pliegue de entrenamiento y, al final, sobre el entrenamiento completo. Nunca ve la prueba.
- El umbral se elige con las probabilidades *out-of-fold* del entrenamiento, de modo que el modelo señale la misma proporción que la regla en entrenamiento. Ese umbral se aplica después a la prueba.
- La comparación con la regla usa el conjunto de prueba. La diferencia de *recall* lleva un intervalo bootstrap del 95 % (2.000 remuestras). El criterio de éxito de P1 es que el límite inferior no baje de −0,05.
- La regla es la aprobada: primer semestre, menos de la mitad de las unidades aprobadas. No hay una regla distinta para E3.

## 0. Configuración

In [1]:
# 0.1 Librerías y rutas
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_BOOT = 2000
MARGEN = -0.05

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
DATA = ROOT / "TFM2" / "Datos" / "predict+students+dropout+and+academic+success" / "data.csv"
RESULTADOS = ROOT / "resultados"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Datos, partición y regla

La regla se recalcula con la misma definición del cuaderno 02, y se comprueba que la proporción de señalados en prueba coincide con la ya guardada.

In [2]:
# 1.1 Carga y partición guardada
df = pd.read_csv(DATA, sep=";", encoding="utf-8-sig")
df.columns = df.columns.str.strip()
df["dropout"] = (df["Target"] == "Dropout").astype(int)

idx_train = pd.read_csv(RESULTADOS / "idx_train.csv")["idx"].to_numpy()
idx_test = pd.read_csv(RESULTADOS / "idx_test.csv")["idx"].to_numpy()
assert len(set(idx_train) & set(idx_test)) == 0
assert len(idx_train) + len(idx_test) == len(df)

A1, M1 = "Curricular units 1st sem (approved)", "Curricular units 1st sem (enrolled)"
sin = df[M1] == 0
df["regla"] = ((df[A1] / df[M1].where(~sin) < 0.5) | sin).astype(int)

train = df.loc[idx_train]
test = df.loc[idx_test]
TASA_REGLA = float(train["regla"].mean())
print(f"Entrenamiento {len(train)} | prueba {len(test)}")
print(f"Regla en entrenamiento: {TASA_REGLA:.1%} señalados | recall {train.loc[train['dropout']==1, 'regla'].mean():.3f}")
print(f"Regla en prueba: {test['regla'].mean():.1%} señalados | recall {recall_score(test['dropout'], test['regla']):.3f}")

Entrenamiento 3539 | prueba 885
Regla en entrenamiento: 23.3% señalados | recall 0.555
Regla en prueba: 22.9% señalados | recall 0.581


## 2. Escenarios

Los grupos son los de `01_eda_dataset`. E1 deja fuera deudor, matrícula al día y becario. E1+ los incorpora. E2 añade el primer semestre y E3 el segundo.

In [3]:
# 2.1 Predictores por escenario
sem1 = [c for c in df.columns if c.startswith("Curricular units 1st sem")]
sem2 = [c for c in df.columns if c.startswith("Curricular units 2nd sem")]
financieras = ["Debtor", "Tuition fees up to date", "Scholarship holder"]
matricula = [
    "Marital status", "Nacionality", "Displaced", "Gender", "Age at enrollment", "International",
    "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation",
    "Educational special needs", *financieras,
    "Unemployment rate", "Inflation rate", "GDP",
    "Application mode", "Application order", "Course", "Daytime/evening attendance",
    "Previous qualification", "Previous qualification (grade)", "Admission grade",
]
ESCENARIOS = {
    "E1": [c for c in matricula if c not in financieras],
    "E1+": matricula,
    "E2": matricula + sem1,
    "E3": matricula + sem1 + sem2,
}
for nombre, cols in ESCENARIOS.items():
    assert set(cols) <= set(df.columns)
print({k: len(v) for k, v in ESCENARIOS.items()})

{'E1': 21, 'E1+': 24, 'E2': 30, 'E3': 36}


## 3. Modelos y umbral

Dos ensambles, con hiperparámetros fijos y documentados. La validación de cinco pliegues no elige hiperparámetros: elige el umbral. Así la prueba no interviene ni en el ajuste ni en el punto de corte.

SMOTE sobre códigos enteros puede generar valores no enteros. Se mantiene porque el protocolo lo fija y porque los árboles no exigen códigos enteros. Queda anotado como limitación.

In [4]:
# 3.1 Definición
MODELOS = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=SEED, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.1, max_depth=6, random_state=SEED
    ),
}

def umbral_por_proporcion(proba, proporcion):
    """Umbral que señala, como máximo, la proporción indicada (los de mayor probabilidad)."""
    proba = np.asarray(proba)
    if proporcion <= 0:
        return np.nextafter(proba.max(), np.inf)
    corte = np.quantile(proba, 1 - proporcion)
    return float(corte)

def entrenar_y_predecir(modelo, X_train, y_train, X_test, proporcion):
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X_train))
    for tr, va in cv.split(X_train, y_train):
        X_tr, y_tr = SMOTE(random_state=SEED).fit_resample(X_train.iloc[tr], y_train.iloc[tr])
        m = clone(modelo)
        m.fit(X_tr, y_tr)
        oof[va] = m.predict_proba(X_train.iloc[va])[:, 1]
    umbral = umbral_por_proporcion(oof, proporcion)
    X_tr, y_tr = SMOTE(random_state=SEED).fit_resample(X_train, y_train)
    final = clone(modelo)
    final.fit(X_tr, y_tr)
    proba = final.predict_proba(X_test)[:, 1]
    return (proba >= umbral).astype(int), umbral, final

## 4. Ajuste y evaluación en prueba

In [5]:
# 4.1 Entrenar los ocho modelos
y_train = train["dropout"]
y_test = test["dropout"].to_numpy()
regla_test = test["regla"].to_numpy()
filas = []
predicciones = {"y": y_test, "regla": regla_test}

for escenario, cols in ESCENARIOS.items():
    X_train = train[cols]
    X_test = test[cols]
    for nombre, modelo in MODELOS.items():
        pred, umbral, _ = entrenar_y_predecir(modelo, X_train, y_train, X_test, TASA_REGLA)
        clave = f"{nombre} | {escenario}"
        predicciones[clave] = pred
        filas.append({
            "modelo": nombre,
            "escenario": escenario,
            "predictores": len(cols),
            "umbral": round(umbral, 4),
            "% señalados": round(100 * pred.mean(), 1),
            "recall": round(recall_score(y_test, pred, zero_division=0), 3),
            "precisión": round(precision_score(y_test, pred, zero_division=0), 3),
            "F1": round(f1_score(y_test, pred, zero_division=0), 3),
        })
        print(clave, "listo")

tabla = pd.DataFrame(filas)
tabla

Random Forest | E1 listo


HistGradientBoosting | E1 listo


Random Forest | E1+ listo


HistGradientBoosting | E1+ listo


Random Forest | E2 listo


HistGradientBoosting | E2 listo


Random Forest | E3 listo


HistGradientBoosting | E3 listo


,modelo,escenario,predictores,umbral,% señalados,recall,precisión,F1
0,Random Forest,E1,21,0.5376,23.7,0.447,0.605,0.514
1,HistGradientBoosting,E1,21,0.5493,23.3,0.444,0.612,0.514
2,Random Forest,E1+,24,0.5196,23.3,0.521,0.718,0.604
3,HistGradientBoosting,E1+,24,0.5532,23.3,0.532,0.733,0.616
4,Random Forest,E2,30,0.6297,23.8,0.655,0.882,0.752
5,HistGradientBoosting,E2,30,0.7065,24.4,0.676,0.889,0.768
6,Random Forest,E3,36,0.6674,23.2,0.669,0.927,0.777
7,HistGradientBoosting,E3,36,0.7681,23.7,0.676,0.914,0.777


## 5. Diferencia de recall frente a la regla

Intervalo bootstrap emparejado: en cada remuestra se calcula el *recall* del modelo y el de la regla sobre los mismos estudiantes. P1 se cumple si, en E1, el límite inferior del intervalo de la diferencia es mayor o igual que −0,05.

In [6]:
# 5.1 Bootstrap de la diferencia
rng = np.random.default_rng(SEED)
diffs = {}
for clave, pred in predicciones.items():
    if clave in ("y", "regla"):
        continue
    muestras = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y_test), len(y_test))
        muestras.append(
            recall_score(y_test[i], pred[i], zero_division=0)
            - recall_score(y_test[i], regla_test[i], zero_division=0)
        )
    diffs[clave] = muestras

comp = tabla.copy()
comp["diferencia recall"] = [
    round(recall_score(y_test, predicciones[f"{r.modelo} | {r.escenario}"], zero_division=0)
          - recall_score(y_test, regla_test), 3)
    for r in comp.itertuples()
]
comp["IC 95 % inf."] = [round(np.percentile(diffs[f"{r.modelo} | {r.escenario}"], 2.5), 3) for r in comp.itertuples()]
comp["IC 95 % sup."] = [round(np.percentile(diffs[f"{r.modelo} | {r.escenario}"], 97.5), 3) for r in comp.itertuples()]
comp["P1 (límite ≥ −0,05)"] = np.where(comp["escenario"].eq("E1"), comp["IC 95 % inf."] >= MARGEN, "—")
comp

,modelo,escenario,predictores,umbral,% señalados,recall,precisión,F1,diferencia recall,IC 95 % inf.,IC 95 % sup.,"P1 (límite ≥ −0,05)"
0,Random Forest,E1,21,0.5376,23.7,0.447,0.605,0.514,-0.134,-0.207,-0.064,False
1,HistGradientBoosting,E1,21,0.5493,23.3,0.444,0.612,0.514,-0.137,-0.215,-0.062,False
2,Random Forest,E1+,24,0.5196,23.3,0.521,0.718,0.604,-0.060,-0.131,0.010,—
3,HistGradientBoosting,E1+,24,0.5532,23.3,0.532,0.733,0.616,-0.049,-0.124,0.026,—
4,Random Forest,E2,30,0.6297,23.8,0.655,0.882,0.752,0.074,0.018,0.126,—
5,HistGradientBoosting,E2,30,0.7065,24.4,0.676,0.889,0.768,0.095,0.035,0.154,—
6,Random Forest,E3,36,0.6674,23.2,0.669,0.927,0.777,0.088,0.031,0.144,—
7,HistGradientBoosting,E3,36,0.7681,23.7,0.676,0.914,0.777,0.095,0.036,0.159,—


## 6. P2: qué aporta cada semestre

Misma tabla, ordenada por escenario, para ver si el *recall* sube de E1 a E2 y de E2 a E3. La regla no cambia: la comparación es contra la regla del primer semestre.

In [7]:
# 6.1 Recall por escenario
comp.pivot(index="modelo", columns="escenario", values="recall")[["E1", "E1+", "E2", "E3"]]

escenario,E1,E1+,E2,E3
modelo,,,,
HistGradientBoosting,0.444,0.532,0.676,0.676
Random Forest,0.447,0.521,0.655,0.669


## 6.2 Proporción necesaria para igualar el recall de la regla

Misma puntuación de E1, otro punto de corte. Se calcula qué proporción de estudiantes hay que señalar para alcanzar el recall de la regla: en las puntuaciones *out-of-fold* del entrenamiento, el recall de la regla en entrenamiento; en la prueba, el recall de la regla en la prueba. No se usa para elegir el modelo ni el umbral del protocolo.


In [8]:
# 6.2 Tasa de señalados para igualar el recall de la regla
def tasa_para_recall(y, proba, recall_objetivo):
    y = np.asarray(y)
    proba = np.asarray(proba)
    orden = np.argsort(-proba)
    capturados = np.cumsum(y[orden])
    total = capturados[-1]
    k = int(np.searchsorted(capturados, recall_objetivo * total, side="left")) + 1
    k = min(k, len(y))
    return k / len(y), capturados[k - 1] / total

def puntuaciones_e1(modelo, cols):
    X_train, y_train = train[cols], train["dropout"]
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X_train))
    for tr, va in cv.split(X_train, y_train):
        X_tr, y_tr = SMOTE(random_state=SEED).fit_resample(X_train.iloc[tr], y_train.iloc[tr])
        m = clone(modelo)
        m.fit(X_tr, y_tr)
        oof[va] = m.predict_proba(X_train.iloc[va])[:, 1]
    X_tr, y_tr = SMOTE(random_state=SEED).fit_resample(X_train, y_train)
    final = clone(modelo)
    final.fit(X_tr, y_tr)
    return oof, final.predict_proba(test[cols])[:, 1]

recall_train = recall_score(train["dropout"], train["regla"])
recall_test = recall_score(test["dropout"], test["regla"])
filas_tasa = []
for nombre, modelo in MODELOS.items():
    oof, proba = puntuaciones_e1(modelo, ESCENARIOS["E1"])
    t_tr, _ = tasa_para_recall(train["dropout"], oof, recall_train)
    t_te, _ = tasa_para_recall(test["dropout"], proba, recall_test)
    filas_tasa.append({
        "modelo": nombre,
        "recall regla, entrenamiento": round(recall_train, 3),
        "% a señalar en entrenamiento": round(100 * t_tr, 1),
        "recall regla, prueba": round(recall_test, 3),
        "% a señalar en prueba": round(100 * t_te, 1),
    })
tasa_igualar = pd.DataFrame(filas_tasa)
tasa_igualar.to_csv(RESULTADOS / "e1_tasa_para_igualar_recall.csv", index=False, encoding="utf-8")
tasa_igualar


,modelo,"recall regla, entrenamiento",% a señalar en entrenamiento,"recall regla, prueba",% a señalar en prueba
0,Random Forest,0.555,33.3,0.581,34.7
1,HistGradientBoosting,0.555,33.0,0.581,34.8


## 7. Guardar

In [9]:
# 7.1 Exportar
comp.to_csv(RESULTADOS / "modelos_prueba.csv", index=False, encoding="utf-8")
print(RESULTADOS / "modelos_prueba.csv")

/Users/valentina.chacon/dev/TFM/resultados/modelos_prueba.csv


## 8. Lectura

Las cifras de las celdas 5.1 y 6.1 son las que pasan al capítulo 5. Este cuaderno no las interpreta: la discusión va en el capítulo 6.